Jeremy Granflaten 2/22/2026

Need to recreate the MLFlow example from the text and the code provided. I had to create a new environment and install new packages for mlflow, tiktoken, and prometheus-client.

Made sure I imported all the libraries I would need and the API key

In [2]:
import os
import time
import mlflow
import tiktoken as tk

from openai import OpenAI
from dotenv import load_dotenv

In [6]:
load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=API_KEY)



OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable


Making sure the tracking URI is correct, since I had a localhost in there before, and that didn't work. I had to verify what I needed in Anaconda Prompt. Store each prompt as a separate run. MLflow to track what is happening and save the results for later comparison. 

In [12]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")

mlflow.set_experiment("GenAI_MLflow_Assignment")

mlflow.autolog()

print("MLflow Connected")

2026/02/19 21:18:30 INFO mlflow.tracking.fluent: Autologging successfully enabled for openai.


MLflow Connected



Just like in the code given, going to track the token function. This would help assess performance, costs, and prompt efficiency. This will help evaluate the enerative AI system and how it is working.

In [18]:

def count_tokens(text, encoding_name="cl100k_base"):
    encoding = tk.get_encoding(encoding_name)
    return len(encoding.encode(text, disallowed_special=()))


Getting the model parameters and using gpt-4o-mini, since it is newer than the one used in the code provided, and should run faster. I chose a temperature of 0.7. I think it gives a good balance between creativity and accuracy. I didn't want responses that were too unpredictable. I set it at 500 tokens so the responses were not too long and made the responses more consistent.

In [22]:
MODEL = "gpt-4o-mini"

TEMPERATURE = 0.7

MAX_TOKENS = 500

Came up with 4 AI-related prompts to see how responses come back and the details of how these are answered. Each prompt will be tracked separately in Mflow to know how the token, latency, and response quality vary across prompts. Each of these prompts will help evaluate the model's performance. Since they are all AI-related, you can really see how the prompts perform.

In [25]:

prompts = [

    "Explain what Artificial Intelligence is.",

    "Explain Machine Learning in simple terms.",

    "What are the ethical concerns of AI?",

    "How will AI impact jobs in the future?"

]

Each prompt is run separately so that MLflow can track each individually. This helps me look at each response for how long it took, how many tokens it had, and the response itself. This can help improve my prompts, which I changed a couple of times, and added the last prompt since it is something everyone is worried about.


In [27]:

for prompt in prompts:

    with mlflow.start_run():

        print("\nPROMPT:")
        print(prompt)

        start_time = time.time()

        response = client.chat.completions.create(

            model=MODEL,

            messages=[

                {"role": "system", "content": "You are a helpful assistant."},

                {"role": "user", "content": prompt}

            ],

            temperature=TEMPERATURE,

            max_tokens=MAX_TOKENS

        )

        latency = time.time() - start_time

        output = response.choices[0].message.content

        print("\nRESPONSE:")
        print(output)

        prompt_tokens = count_tokens(prompt)

        completion_tokens = count_tokens(output)

        total_tokens = prompt_tokens + completion_tokens


        mlflow.log_param("prompt", prompt)

        mlflow.log_param("model", MODEL)

        mlflow.log_param("temperature", TEMPERATURE)


        mlflow.log_metric("request_latency", latency)

        mlflow.log_metric("prompt_tokens", prompt_tokens)

        mlflow.log_metric("completion_tokens", completion_tokens)

        mlflow.log_metric("total_tokens", total_tokens)

        mlflow.log_metric("request_count", 1)


PROMPT:
Explain what Artificial Intelligence is.

RESPONSE:
Artificial Intelligence (AI) refers to the simulation of human intelligence processes by machines, particularly computer systems. It encompasses a wide range of technologies and methodologies that enable machines to perform tasks that typically require human intelligence. These tasks include:

1. **Learning**: The ability of AI systems to improve their performance over time by processing data and identifying patterns. This is often achieved through techniques such as machine learning, where algorithms are trained on large datasets to recognize patterns and make predictions.

2. **Reasoning**: The capability to solve problems and make decisions based on available information. This involves logical reasoning, understanding complex scenarios, and drawing conclusions.

3. **Perception**: The ability to interpret sensory information from the environment. This includes computer vision (analyzing images and videos), natural language

[Trace(trace_id=tr-9aecc1338f097f812b7b7e8b88ddcebd), Trace(trace_id=tr-8e42ea875e3617e5b8765120f8ae22eb), Trace(trace_id=tr-2cbfb7584d0e733df14d38270ae90045), Trace(trace_id=tr-0d679ebdaa6f311a7ce9f76fa4536ba0)]

When looking at each prompt, the first 2 questions had the lower tokens, and the first prompt had the lowest latency by a wide margin. The last question had the most tokens and the most latency. I think this is because it is more of a hypothetical question than the other 3 questions. The other 3 questions are more of a summary of information that you can find. The prompt about machine learning had the lowest score across everything, which is something to think about and why it would be so much lower in tokens and latency.